# Player Ability

This script is my attempt at feature engineering to find different, reliable ways of quantifying the player's scoring ability and the goalkeeper's saving abilities for every shot taken.

This is to perform a shot-level analysis of shooter and goalkeeper effects to ultimately answer the question:
    Does player finishing ability and goalkeeper ability improve shot-level xG beyond geometry/context?

*Uncomment commented cells to stream and get full dataset if you don't have that already*

In [1]:
import pandas as pd

shots = pd.read_csv("~/Desktop/Football_Stats/xG/datasets/processed/Big5_shots.csv")
shots.head()

,location,player,player_id,position,shot_aerial_won,shot_first_time,shot_statsbomb_xg,team,under_pressure,shot_open_goal,...,shot_technique_Overhead Kick,shot_technique_Volley,shot_type_Corner,shot_type_Free Kick,shot_type_Open Play,shot_type_Penalty,x,y,distance,angle
0,"[94.5, 42.9]",Aaron Ramsey,3517.0,Right Wing,0,1,0.038832,Arsenal,0,0,...,0,0,0,0,1,0,94.5,42.9,25.664372,17.611035
1,"[93.5, 48.4]",Francesc Fàbregas i Soler,3478.0,Right Defensive Midfield,0,0,0.031541,Chelsea,1,0,...,0,0,0,0,1,0,93.5,48.4,27.799460,15.648789
2,"[89.8, 22.3]",Alexis Alejandro Sánchez Sánchez,3385.0,Left Wing,0,0,0.006660,Arsenal,1,0,...,0,0,0,0,1,0,89.8,22.3,35.004714,11.297814
3,"[98.2, 26.0]",Diego da Silva Costa,5198.0,Center Forward,0,0,0.022902,Chelsea,0,0,...,0,0,0,0,1,0,98.2,26.0,25.908300,14.904419
4,"[105.7, 24.2]",Theo Walcott,3668.0,Center Forward,0,0,0.049741,Arsenal,0,0,...,0,0,0,0,1,0,105.7,24.2,21.310326,14.633756


In [2]:
shots.position.unique()

array(['Right Wing', 'Right Defensive Midfield', 'Left Wing',
       'Center Forward', 'Right Center Back', 'Right Back',
       'Left Defensive Midfield', 'Center Attacking Midfield',
       'Left Center Back', 'Left Back', 'Left Center Midfield',
       'Right Center Midfield', 'Center Defensive Midfield',
       'Right Midfield', 'Left Midfield', 'Right Wing Back',
       'Left Wing Back', 'Right Center Forward', 'Left Center Forward',
       'Center Back', 'Left Attacking Midfield',
       'Right Attacking Midfield', 'Goalkeeper'], dtype=object)

In [3]:
shots.player.value_counts().sort_values()

player
Mirko Valdifiori                         1
Gary Hooper                              1
Eunan O'Kane                             1
Marvin Emnes                             1
Seydou Doumbia                           1
                                      ... 
Zlatan Ibrahimović                     148
Harry Kane                             158
Lionel Andrés Messi Cuccittini         158
Gonzalo Gerardo Higuaín                182
Cristiano Ronaldo dos Santos Aveiro    228
Name: count, Length: 1991, dtype: int64

Classical xG mostly asks:
    Given the shot situation, how likely is this shot to become a goal?

From further investigation, I think I will eventually remove penalties from the dataset. Although the models learn to differentiate scenarios(open player, set pieces and penalties), penalties present much more "constant" features than other shot-types.

They involve fixed geometry, nearly fixed angle, nearly fixed distance, and are pychologically and tactically unique. Not to mention, on further data investigation, the xG of penalties is fixed between ~ 0.75 - 0.80. 

I will perform analysis on the full data, then either seperate the model into penalties and non-penalties and perform analysis on the different datasets, or Keep penalties in dataset with penalty indicator feature.


In [4]:
shots.columns

Index(['location', 'player', 'player_id', 'position', 'shot_aerial_won',
       'shot_first_time', 'shot_statsbomb_xg', 'team', 'under_pressure',
       'shot_open_goal', 'shot_follows_dribble', 'goalkeeper_success_out',
       'half_end_early_video_end', 'goal', 'shot_body_part_Head',
       'shot_body_part_Left Foot', 'shot_body_part_Other',
       'shot_body_part_Right Foot', 'shot_technique_Backheel',
       'shot_technique_Diving Header', 'shot_technique_Half Volley',
       'shot_technique_Lob', 'shot_technique_Normal',
       'shot_technique_Overhead Kick', 'shot_technique_Volley',
       'shot_type_Corner', 'shot_type_Free Kick', 'shot_type_Open Play',
       'shot_type_Penalty', 'x', 'y', 'distance', 'angle'],
      dtype='object')

For the player/goalkeeper ability ratings, I will use a dataset, scraped by (https://www.kaggle.com/datasets/yarknyorulmaz/fifa-index-player-ratings-dataset-epl-v16-v24), which contain detailed player statistics and attributes for football players as featured in the FIFAIndex database, which is a widely recognized source of football player data used for simulation and analysis purposes. 

In [5]:
df = pd.read_csv("~/Desktop/Football_Stats/xG/datasets/raw/players_16.csv")
df.head()

/var/folders/lz/th4y3_7n34lfmhmw0b_t5y0r0000gn/T/ipykernel_76444/3489828024.py:1: DtypeWarning: Columns (104) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("~/Desktop/Football_Stats/xG/datasets/raw/players_16.csv")


,sofifa_id,player_url,short_name,long_name,player_positions,overall,potential,value_eur,wage_eur,age,...,lcb,cb,rcb,rb,gk,player_face_url,club_logo_url,club_flag_url,nation_logo_url,nation_flag_url
0,158023,https://sofifa.com/player/158023/lionel-messi/...,L. Messi,Lionel Andrés Messi Cuccittini,"RW, CF",94,95,111000000.0,550000.0,28,...,44+3,44+3,44+3,57+3,19+3,https://cdn.sofifa.net/players/158/023/16_120.png,https://cdn.sofifa.net/teams/241/60.png,https://cdn.sofifa.net/flags/es.png,https://cdn.sofifa.net/teams/1369/60.png,https://cdn.sofifa.net/flags/ar.png
1,20801,https://sofifa.com/player/20801/c-ronaldo-dos-...,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,"LW, LM",93,93,85500000.0,475000.0,30,...,52+3,52+3,52+3,60+3,20+3,https://cdn.sofifa.net/players/020/801/16_120.png,https://cdn.sofifa.net/teams/243/60.png,https://cdn.sofifa.net/flags/es.png,https://cdn.sofifa.net/teams/1354/60.png,https://cdn.sofifa.net/flags/pt.png
2,9014,https://sofifa.com/player/9014/arjen-robben/16...,A. Robben,Arjen Robben,"RM, LM, RW",90,90,56000000.0,250000.0,31,...,47+3,47+3,47+3,59+3,19+3,https://cdn.sofifa.net/players/009/014/16_120.png,https://cdn.sofifa.net/teams/21/60.png,https://cdn.sofifa.net/flags/de.png,https://cdn.sofifa.net/teams/105035/60.png,https://cdn.sofifa.net/flags/nl.png
3,167495,https://sofifa.com/player/167495/manuel-neuer/...,M. Neuer,Manuel Peter Neuer,GK,90,90,58000000.0,250000.0,29,...,33+3,33+3,33+3,33+3,87+3,https://cdn.sofifa.net/players/167/495/16_120.png,https://cdn.sofifa.net/teams/21/60.png,https://cdn.sofifa.net/flags/de.png,https://cdn.sofifa.net/teams/1337/60.png,https://cdn.sofifa.net/flags/de.png
4,176580,https://sofifa.com/player/176580/luis-suarez/1...,L. Suárez,Luis Alberto Suárez Díaz,ST,90,90,69000000.0,300000.0,28,...,58+3,58+3,58+3,64+3,37+3,https://cdn.sofifa.net/players/176/580/16_120.png,https://cdn.sofifa.net/teams/241/60.png,https://cdn.sofifa.net/flags/es.png,NaN,https://cdn.sofifa.net/flags/uy.png


In [6]:
df.columns

Index(['sofifa_id', 'player_url', 'short_name', 'long_name',
       'player_positions', 'overall', 'potential', 'value_eur', 'wage_eur',
       'age',
       ...
       'lcb', 'cb', 'rcb', 'rb', 'gk', 'player_face_url', 'club_logo_url',
       'club_flag_url', 'nation_logo_url', 'nation_flag_url'],
      dtype='object', length=110)

In [7]:
# We want to focus our analysis now on infield players
df = df[~df['player_positions'].str.contains('GK', case=False)]
df.head()

,sofifa_id,player_url,short_name,long_name,player_positions,overall,potential,value_eur,wage_eur,age,...,lcb,cb,rcb,rb,gk,player_face_url,club_logo_url,club_flag_url,nation_logo_url,nation_flag_url
0,158023,https://sofifa.com/player/158023/lionel-messi/...,L. Messi,Lionel Andrés Messi Cuccittini,"RW, CF",94,95,111000000.0,550000.0,28,...,44+3,44+3,44+3,57+3,19+3,https://cdn.sofifa.net/players/158/023/16_120.png,https://cdn.sofifa.net/teams/241/60.png,https://cdn.sofifa.net/flags/es.png,https://cdn.sofifa.net/teams/1369/60.png,https://cdn.sofifa.net/flags/ar.png
1,20801,https://sofifa.com/player/20801/c-ronaldo-dos-...,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,"LW, LM",93,93,85500000.0,475000.0,30,...,52+3,52+3,52+3,60+3,20+3,https://cdn.sofifa.net/players/020/801/16_120.png,https://cdn.sofifa.net/teams/243/60.png,https://cdn.sofifa.net/flags/es.png,https://cdn.sofifa.net/teams/1354/60.png,https://cdn.sofifa.net/flags/pt.png
2,9014,https://sofifa.com/player/9014/arjen-robben/16...,A. Robben,Arjen Robben,"RM, LM, RW",90,90,56000000.0,250000.0,31,...,47+3,47+3,47+3,59+3,19+3,https://cdn.sofifa.net/players/009/014/16_120.png,https://cdn.sofifa.net/teams/21/60.png,https://cdn.sofifa.net/flags/de.png,https://cdn.sofifa.net/teams/105035/60.png,https://cdn.sofifa.net/flags/nl.png
4,176580,https://sofifa.com/player/176580/luis-suarez/1...,L. Suárez,Luis Alberto Suárez Díaz,ST,90,90,69000000.0,300000.0,28,...,58+3,58+3,58+3,64+3,37+3,https://cdn.sofifa.net/players/176/580/16_120.png,https://cdn.sofifa.net/teams/241/60.png,https://cdn.sofifa.net/flags/es.png,NaN,https://cdn.sofifa.net/flags/uy.png
5,41236,https://sofifa.com/player/41236/zlatan-ibrahim...,Z. Ibrahimović,Zlatan Ibrahimović,ST,89,89,40500000.0,220000.0,33,...,53+3,53+3,53+3,56+3,20+3,https://cdn.sofifa.net/players/041/236/16_120.png,https://cdn.sofifa.net/teams/73/60.png,https://cdn.sofifa.net/flags/fr.png,https://cdn.sofifa.net/teams/1363/60.png,https://cdn.sofifa.net/flags/se.png


I am including "preferred_foot" and "weak_foot" to account for the player's foot preference, which can influence their scoring ability. I am including "attacking_heading_accuracy" since some scoring contexts are aerial won, meaning the player's heading accuracy is necessary to accurately capture their scoring ability in aerial situations

In [8]:
cols = ['sofifa_id', 'player_url', 'short_name', 'long_name', 'player_positions', 'club_name', 'shooting', 'attacking_heading_accuracy', 'preferred_foot', 'weak_foot']
df = df[cols].copy()

# df['shooting'] = (df['shooting'] + df['attacking_heading_accuracy']) / 2
# df.drop(columns=['attacking_heading_accuracy'], inplace=True)

df.head()

,sofifa_id,player_url,short_name,long_name,player_positions,club_name,shooting,attacking_heading_accuracy,preferred_foot,weak_foot
0,158023,https://sofifa.com/player/158023/lionel-messi/...,L. Messi,Lionel Andrés Messi Cuccittini,"RW, CF",FC Barcelona,88.0,71,Left,4
1,20801,https://sofifa.com/player/20801/c-ronaldo-dos-...,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,"LW, LM",Real Madrid CF,93.0,86,Right,4
2,9014,https://sofifa.com/player/9014/arjen-robben/16...,A. Robben,Arjen Robben,"RM, LM, RW",FC Bayern München,86.0,51,Left,2
4,176580,https://sofifa.com/player/176580/luis-suarez/1...,L. Suárez,Luis Alberto Suárez Díaz,ST,FC Barcelona,88.0,77,Right,4
5,41236,https://sofifa.com/player/41236/zlatan-ibrahim...,Z. Ibrahimović,Zlatan Ibrahimović,ST,Paris Saint-Germain,90.0,76,Right,4


In [9]:
df["source"] = "stefanoleone992/fifa-22-complete-player-dataset"
df.head()

,sofifa_id,player_url,short_name,long_name,player_positions,club_name,shooting,attacking_heading_accuracy,preferred_foot,weak_foot,source
0,158023,https://sofifa.com/player/158023/lionel-messi/...,L. Messi,Lionel Andrés Messi Cuccittini,"RW, CF",FC Barcelona,88.0,71,Left,4,stefanoleone992/fifa-22-complete-player-dataset
1,20801,https://sofifa.com/player/20801/c-ronaldo-dos-...,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,"LW, LM",Real Madrid CF,93.0,86,Right,4,stefanoleone992/fifa-22-complete-player-dataset
2,9014,https://sofifa.com/player/9014/arjen-robben/16...,A. Robben,Arjen Robben,"RM, LM, RW",FC Bayern München,86.0,51,Left,2,stefanoleone992/fifa-22-complete-player-dataset
4,176580,https://sofifa.com/player/176580/luis-suarez/1...,L. Suárez,Luis Alberto Suárez Díaz,ST,FC Barcelona,88.0,77,Right,4,stefanoleone992/fifa-22-complete-player-dataset
5,41236,https://sofifa.com/player/41236/zlatan-ibrahim...,Z. Ibrahimović,Zlatan Ibrahimović,ST,Paris Saint-Germain,90.0,76,Right,4,stefanoleone992/fifa-22-complete-player-dataset


In [10]:
df['long_name'].value_counts().sort_values()

long_name
Jairo Izquierdo González    1
Andrea Barzagli             1
Diego da Silva Costa        1
Karim Benzema               1
Daniel Alves da Silva       1
                           ..
Mohamed Fofana              2
Reece Brown                 2
Dominic Thomas              2
Lasse Nielsen               2
Marco Rojas                 2
Name: count, Length: 13907, dtype: int64

## Mapping Pipeline

From my inspection, player names are mostly similarly represented on the StatsBomb dataset and the SoFIFA dataset. However, there are player's that are represented differently. Indeed there are two players called Reece Brown, Dominic Thomas, Lasse Nielsen, etc. 

Validate dictionaries
        ↓
Normalize player names
        ↓
Map StatsBomb teams → SoFIFA club names
        ↓
Map StatsBomb positions → SoFIFA positions
        ↓
Aggregate StatsBomb rows by player_id
        ↓
Exact name match
        │
        ├── 0 → UNMATCHED
        │
        ├── 1 → ACCEPT
        │
        └── >1
              ↓
           Team match
              │
              ├── 1 → ACCEPT
              │
              ├── 0 → MANUAL REVIEW
              │
              └── >1
                    ↓
                 Position overlap
                    │
                    ├── 1 → ACCEPT
                    │
                    └── otherwise
                           ↓
                       MANUAL REVIEW

Step 1: Team and Position Mapping

In [11]:
# df['player_positions'].unique()

sofifa_position_codes = set()
for value in df["player_positions"].dropna():
    positions = [pos.strip() for pos in value.split(",")]
    sofifa_position_codes.update(positions)

print(sofifa_position_codes)

# df['club_name'].unique()

{'RM', 'CAM', 'RWB', 'CF', 'CM', 'CB', 'ST', 'LB', 'RB', 'LW', 'LM', 'LWB', 'CDM', 'RW'}


In [12]:
shots.position.unique()
# shots.team.unique()

array(['Right Wing', 'Right Defensive Midfield', 'Left Wing',
       'Center Forward', 'Right Center Back', 'Right Back',
       'Left Defensive Midfield', 'Center Attacking Midfield',
       'Left Center Back', 'Left Back', 'Left Center Midfield',
       'Right Center Midfield', 'Center Defensive Midfield',
       'Right Midfield', 'Left Midfield', 'Right Wing Back',
       'Left Wing Back', 'Right Center Forward', 'Left Center Forward',
       'Center Back', 'Left Attacking Midfield',
       'Right Attacking Midfield', 'Goalkeeper'], dtype=object)

In [13]:
# One-to-One Mapping
# position_map = {"Right Wing": "RW",
#                 "Left Wing": "LW",

#                 "Center Forward": "ST",
#                 "Right Center Forward": "CF",
#                 "Left Center Forward": "CF",

#                 "Center Attacking Midfield": "CAM",
#                 "Left Attacking Midfield": "CAM",
#                 "Right Attacking Midfield": "CAM",

#                 "Center Defensive Midfield": "CDM",
#                 "Right Defensive Midfield": "CDM",
#                 "Left Defensive Midfield": "CDM",

#                 "Right Center Midfield": "CM",
#                 "Left Center Midfield": "CM",

#                 "Right Midfield": "RM",
#                 "Left Midfield": "LM",

#                 "Right Wing Back": "RWB",
#                 "Left Wing Back": "LWB",

#                 "Right Center Back": "CB",
#                 "Left Center Back": "CB",
#                 "Center Back": "CB",

#                 "Right Back": "RB",
#                 "Left Back": "LB" }

# # One-to-many mapping
# position_map = {"Right Wing": {"RW", "RM"},
#                 "Left Wing": {"LW", "LM"},
#                 "Center Forward": {"ST", "CF"},
#                 "Right Center Forward": {"CF", "ST", "RW"},
#                 "Left Center Forward": {"CF", "ST", "LW"},
#                 "Center Attacking Midfield": {"CAM", "CM"},
#                 "Right Attacking Midfield": {"CAM", "RM"},
#                 "Left Attacking Midfield": {"CAM", "LM"},
#                 "Center Defensive Midfield": {"CDM", "CM"},
#                 "Right Defensive Midfield": {"CDM", "CM"},
#                 "Left Defensive Midfield": {"CDM", "CM"},
#                 "Right Center Midfield": {"CM", "RM"},
#                 "Left Center Midfield": {"CM", "LM"},
#                 "Right Midfield": {"RM", "RW"},
#                 "Left Midfield": {"LM", "LW"},
#                 "Right Wing Back": {"RWB", "RB", "RM"},
#                 "Left Wing Back": {"LWB", "LB", "LM"},
#                 "Right Center Back": {"CB", "RB"},
#                 "Left Center Back": {"CB", "LB"},
#                 "Center Back": {"CB"},
#                 "Right Back": {"RB", "RWB"},
#                 "Left Back": {"LB", "LWB"},}

# position_map = {"Right Wing": {"RW"}, 
#                 "Left Wing": {"LW"},

#                 "Center Forward": {"ST", "CF"},
#                 "Right Center Forward": {"CF", "ST"},
#                 "Left Center Forward": {"CF", "ST"},

#                 "Center Attacking Midfield": {"CAM"},
#                 "Left Attacking Midfield": {"CAM"},
#                 "Right Attacking Midfield": {"CAM"},

#                 "Center Defensive Midfield": {"CDM"},
#                 "Right Defensive Midfield": {"CDM"},
#                 "Left Defensive Midfield": {"CDM"},

#                 "Right Center Midfield": {"CM"},
#                 "Left Center Midfield": {"CM"},

#                 "Right Midfield": {"RM"},
#                 "Left Midfield": {"LM"},

#                 "Right Wing Back": {"RWB"},
#                 "Left Wing Back": {"LWB"},

#                 "Right Center Back": {"CB"},
#                 "Left Center Back": {"CB"},
#                 "Center Back": {"CB"},

#                 "Right Back": {"RB"},
#                 "Left Back": {"LB"}}

In [14]:
# StatsBomb to SoFIFA
position_map = {"Right Wing": {"RW"}, 
                "Left Wing": {"LW"},
                "Center Forward": {"ST", "CF"},
                "Right Center Forward": {"CF", "ST"},
                "Left Center Forward": {"CF", "ST"},
                "Center Attacking Midfield": {"CAM"},
                "Left Attacking Midfield": {"CAM"},
                "Right Attacking Midfield": {"CAM"},
                "Center Defensive Midfield": {"CDM"},
                "Right Defensive Midfield": {"CDM"},
                "Left Defensive Midfield": {"CDM"},
                "Right Center Midfield": {"CM"},
                "Left Center Midfield": {"CM"},
                "Right Midfield": {"RM"},
                "Left Midfield": {"LM"},
                "Right Wing Back": {"RWB"},
                "Left Wing Back": {"LWB"},
                "Right Center Back": {"CB"},
                "Left Center Back": {"CB"},
                "Center Back": {"CB"},
                "Right Back": {"RB"},
                "Left Back": {"LB"}}

# StatsBomb to SoFIFA
team_map = {'Arsenal': 'Arsenal', 'Chelsea': 'Chelsea', 'Aston Villa': 'Aston Villa', 'Manchester City': 'Manchester City',
       'Swansea City':'Swansea City' , 'Sunderland':'Sunderland', 'Liverpool':'Liverpool', 'Southampton':'Southampton',
       'AFC Bournemouth':'AFC Bournemouth', 'Leicester City':'Leicester City', 'West Bromwich Albion':'West Bromwich Albion',
       'Newcastle United':'Newcastle United', 'Everton':'Everton', 'Crystal Palace':'Crystal Palace', 'Watford':'Watford',
       'Tottenham Hotspur':'Tottenham Hotspur', 'Norwich City':'Norwich City', 'Stoke City':'Stoke City',
       'Manchester United':'Manchester United', 'West Ham United':'West Ham United', 'Olympique de Marseille':'Olympique de Marseille',
       'Troyes': 'ESTAC Troyes', 'Stade Malherbe Caen': 'Stade Malherbe Caen', 'AS Monaco': 'AS Monaco', 'Bordeaux': 'FC Girondins de Bordeaux',
       'Paris Saint-Germain': 'Paris Saint-Germain', 'Lorient': 'FC Lorient', 'Stade de Reims': 'Stade de Reims',
       'Saint-Étienne': 'AS Saint-Étienne', 'OGC Nice': 'OGC Nice' , 'Toulouse': 'Toulouse Football Club', 'Bastia': 'SC Bastia', 'Guingamp': 'En Avant de Guingamp',
       'Rennes': 'Stade Rennais FC', 'Gazélec Ajaccio': 'GFC Ajaccio', 'Lyon': 'Olympique Lyonnais' , 'Marseille': 'Olympique de Marseille', 'Nantes': 'FC Nantes',
       'Angers': 'Angers SCO', 'Montpellier': 'Montpellier Hérault SC', 'Lille': 'LOSC Lille', 'Caen': 'Stade Malherbe Caen', 'Ingolstadt': 'FC Ingolstadt 04',
       'Bayer Leverkusen': 'Bayer 04 Leverkusen', 'Borussia Mönchengladbach': 'Borussia Mönchengladbach', 'Hertha Berlin': 'Hertha BSC',
       'Schalke 04': 'FC Schalke 04', 'Eintracht Frankfurt': 'Eintracht Frankfurt', 'FC Köln': '1. FC Köln', 'Wolfsburg': 'VfL Wolfsburg',
       'VfB Stuttgart': 'VfB Stuttgart', 'Hamburger SV': 'Hamburger SV', 'Augsburg': 'FC Augsburg', 'Werder Bremen': 'SV Werder Bremen',
       'FSV Mainz 05': '1. FSV Mainz 05', 'Borussia Dortmund': 'Borussia Dortmund', 'Darmstadt 98': 'SV Darmstadt 98',
       'Bayern Munich': 'FC Bayern München', 'Hannover 96': 'Hannover 96', 'Hoffenheim': 'TSG Hoffenheim', 'Sporting Gijón': 'Real Sporting de Gijón',
       'Real Madrid': 'Real Madrid CF', 'Levante UD': 'Levante Unión Deportiva', 'Eibar': 'SD Eibar', 'Las Palmas': 'Unión Deportiva Las Palmas',
       'Sevilla': 'Sevilla FC' , 'RC Deportivo La Coruña': 'Deportivo de La Coruña', 'Getafe': 'Getafe CF', 'Málaga': 'Málaga CF', 
       'Espanyol': 'RCD Espanyol de Barcelona', 'Villarreal': 'Villarreal CF', 'Rayo Vallecano': 'Rayo Vallecano', 'Real Betis': 'Real Betis Balompié' , 
       'Athletic Club': 'Athletic Club de Bilbao', 'Atlético Madrid': 'Atlético de Madrid', 'Celta Vigo': 'RC Celta de Vigo', 'Valencia': 'Valencia CF', 
       'Real Sociedad': 'Real Sociedad', 'Granada': 'Granada CF', 'Barcelona': 'FC Barcelona', 'Hellas Verona': 'Hellas Verona', 'Bologna': 'Bologna', 
       'Lazio': 'Lazio', 'Udinese': 'Udinese Calcio', 'Inter Milan': 'Inter', 'Carpi': 'Carpi', 'Chievo': 'Chievo Verona', 'Genoa': 'Genoa', 
       'AS Roma': 'Roma', 'Sampdoria': 'U.C. Sampdoria', 'Palermo': 'Palermo', 'Atalanta': 'Atalanta', 'Frosinone': 'Frosinone', 'Torino': 'Torino F.C.', 
       'Napoli': 'Napoli', 'Empoli': 'Empoli', 'Fiorentina': 'Fiorentina', 'Sassuolo': 'U.S. Sassuolo Calcio', 'AC Milan': 'AC Milan', 'Juventus': 'Juventus'}

Validating team and player maps

In [15]:
# Team validation
statsbomb_teams = set(shots["team"].dropna().unique())
mapped_statsbomb_teams = set(team_map.keys())
missing_team_keys = statsbomb_teams - mapped_statsbomb_teams # StatsBomb teams that have no entry in team_map
extra_team_keys = mapped_statsbomb_teams - statsbomb_teams # Extra dictionary keys that do not currently occur in StatsBomb

# Check that every mapped SoFIFA club actually exists in df
sofifa_clubs = set(df["club_name"].dropna().unique())

invalid_team_values = {
    sb_team: sofifa_team
    for sb_team, sofifa_team in team_map.items()
    if sofifa_team not in sofifa_clubs
}

print("TEAM MAP VALIDATION")
print("-------------------")
print("StatsBomb teams:", len(statsbomb_teams))
print("Mapped teams:", len(mapped_statsbomb_teams))
print("Missing StatsBomb teams:", missing_team_keys)
print("Extra team-map keys:", extra_team_keys)
print("Mapped SoFIFA clubs not found in df:", invalid_team_values)

# -------------------------------------------------------------------------------------------------------------------------
#  Position validation

# Making sure Goalkeepers are intentionally excluded at this stage
statsbomb_positions = set(shots.loc[shots["position"] != "Goalkeeper", "position"].dropna().unique())
mapped_statsbomb_positions = set(position_map.keys())
missing_position_keys = (statsbomb_positions - mapped_statsbomb_positions)
extra_position_keys = (mapped_statsbomb_positions - statsbomb_positions)

# Validate every code inside every set in position_map
invalid_position_values = {}
for statsbomb_position, mapped_positions in position_map.items():
    invalid_codes = (set(mapped_positions) - sofifa_position_codes)
    if invalid_codes:
        invalid_position_values[statsbomb_position] = invalid_codes

print("\nPOSITION MAP VALIDATION")
print("-----------------------")
print("StatsBomb positions:", len(statsbomb_positions))
print("Missing StatsBomb positions:", missing_position_keys)
print("Extra position-map keys:", extra_position_keys)
print("Mapped codes not found in SoFIFA:", invalid_position_values)

TEAM MAP VALIDATION
-------------------
StatsBomb teams: 100
Mapped teams: 100
Missing StatsBomb teams: set()
Extra team-map keys: set()
Mapped SoFIFA clubs not found in df: {}

POSITION MAP VALIDATION
-----------------------
StatsBomb positions: 22
Missing StatsBomb positions: set()
Extra position-map keys: set()
Mapped codes not found in SoFIFA: {}


In [16]:
# Diagnostic only: SoFIFA positions that are never used anywhere in our mapping
mapped_sofifa_codes = set().union(*position_map.values())
unused_sofifa_codes = (sofifa_position_codes - mapped_sofifa_codes)

print("POSITION MAP VALIDATION")
print("-----------------------")
print(f"StatsBomb outfield positions: {len(statsbomb_positions)}")
print(f"Positions covered by position_map: "
      f"{len(statsbomb_positions & set(position_map.keys()))}")
print(f"Missing StatsBomb positions: {missing_position_keys}")
print(f"Invalid mapped SoFIFA codes: {invalid_position_values}")
print(f"SoFIFA codes unused by position_map: {unused_sofifa_codes}")

# HARD SAFETY CHECKS
assert not missing_team_keys, (
    f"team_map is incomplete. Missing: {missing_team_keys}")

assert not invalid_team_values, (
    f"team_map contains invalid SoFIFA club names: "
    f"{invalid_team_values}" )

assert not missing_position_keys, (
    f"position_map is incomplete. Missing: "
    f"{missing_position_keys}")

assert not invalid_position_values, (
    f"position_map contains invalid SoFIFA position codes: "
    f"{invalid_position_values}")

print("\n✓ All required team and position mappings are valid.")

POSITION MAP VALIDATION
-----------------------
StatsBomb outfield positions: 22
Positions covered by position_map: 22
Missing StatsBomb positions: set()
Invalid mapped SoFIFA codes: {}
SoFIFA codes unused by position_map: set()

✓ All required team and position mappings are valid.


Step 2: Normalize Names

In [17]:
import unicodedata


name_corrections = {"fernando luiz roza": "fernando luiz rosa"}

def normalize_name(name):
    '''returns normalized player names'''
    if pd.isna(name):
        return pd.NA

    name = str(name).lower().strip() # convert to lower case and strip leading and trailing whitespace
    name = (name.replace("’", "'")
                .replace("‘", "'")
                .replace("`", "'")
                .replace("´", "'")
                .replace('"', "'")
                .replace("''", "'")) # Standardize apostrophes first
    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", "ignore").decode("utf-8") # remove accents/diacritics:(non-ASCII characters)
    name = name.replace(".", "")
    name = name.replace("-", " ") # Standardize punctuation/spacing
    name = " ".join(name.split()) # normalize spacing
    name = name_corrections.get(name, name) # Correct known StatsBomb naming inconsistencies

    return name

In [18]:
shots["player_name_norm"] = shots["player"].apply(normalize_name)

df["long_name_norm"] = df["long_name"].apply(normalize_name)
df["short_name_norm"] = df["short_name"].apply(normalize_name)

Normalization Validation and Inspection 

In [19]:
# Validating Normalizing
print("NAME NORMALIZATION VALIDATION")
print("-----------------------------")
print("StatsBomb missing original names:",shots["player"].isna().sum())
print("StatsBomb missing normalized names:",shots["player_name_norm"].isna().sum())
print("SoFIFA missing long names:",df["long_name"].isna().sum())
print("SoFIFA missing normalized long names:",df["long_name_norm"].isna().sum())
print("SoFIFA missing short names:",df["short_name"].isna().sum())
print("SoFIFA missing normalized short names:",df["short_name_norm"].isna().sum())

# Checking for "nan" resulting from normalizng missing values
assert not shots["player_name_norm"].eq("nan").any()
assert not df["long_name_norm"].eq("nan").any()
assert not df["short_name_norm"].eq("nan").any()

print("\n✓ Name normalization completed.")

NAME NORMALIZATION VALIDATION
-----------------------------
StatsBomb missing original names: 0
StatsBomb missing normalized names: 0
SoFIFA missing long names: 0
SoFIFA missing normalized long names: 0
SoFIFA missing short names: 0
SoFIFA missing normalized short names: 0

✓ Name normalization completed.


In [20]:
# Inspecting normalized data relative to original
name_check = pd.DataFrame({"StatsBomb original": shots["player"],
                           "StatsBomb normalized": shots["player_name_norm"]}).drop_duplicates()

name_check.sample( min(10, len(name_check)), random_state=42)

,StatsBomb original,StatsBomb normalized
15751,Valentin Lavigne,valentin lavigne
29032,Marek Hamšík,marek hamsik
5348,Sullay Kaikai,sullay kaikai
19394,Felix Kroos,felix kroos
19546,Kevin Volland,kevin volland
18970,Nicolai Müller,nicolai muller
19841,Daniel Carvajal Ramos,daniel carvajal ramos
22014,David Rodríguez Lombán,david rodriguez lomban
11417,Samuel Yves Umtiti,samuel yves umtiti
29615,Giancarlo González Castro,giancarlo gonzalez castro


In [21]:
# Apply Team and position Mapping
shots["sofifa_club"] = shots["team"].map(team_map)
shots["sofifa_position"] = shots["position"].map(position_map)

In [22]:
# Parse SoFIFA's multiple positions
def parse_sofifa_positions(value):
    """Convert SoFIFA position string into a set."""
    ''' inputs: "ST, LW, RW"
        returns: {"ST", "LW", "RW"} '''

    if pd.isna(value):
        return set()

    return {position.strip() for position in value.split(",")}

df["position_set"] = (df["player_positions"].apply(parse_sofifa_positions))

Step 3: Aggregate StatsBomb rows (by player_id)

In [23]:
def union_position_sets(series):
    """Union the one-to-many SoFIFA position mappings for a player."""
    
    combined = set()
    for value in series.dropna():
        combined.update(value)

    return combined


# Build one StatsBomb identity record per player_id
sb_players = (shots[["player_id", "player", "player_name_norm", "team", "sofifa_club", "position", "sofifa_position"]]
            .groupby(["player_id", "player_name_norm"], dropna=False)
            .agg(original_names=("player", lambda x: set(x.dropna())),
            # Raw StatsBomb values
            teams=("team", lambda x: set(x.dropna())),
            positions=("position", lambda x: set(x.dropna())),
            # SoFIFA-compatible values
            sofifa_teams=("sofifa_club", lambda x: set(x.dropna())),
            sofifa_positions=("sofifa_position", union_position_sets)).reset_index())

sb_players.head(10)

,player_id,player_name_norm,original_names,teams,positions,sofifa_teams,sofifa_positions
0,2936.0,christophe kerbrat,{Christophe Kerbrat},{Guingamp},{Right Center Back},{En Avant de Guingamp},{CB}
1,2943.0,lucas deaux,{Lucas Deaux},{Nantes},"{Left Defensive Midfield, Left Center Back, Ri...",{FC Nantes},"{CDM, CB}"
2,2944.0,benjamin corgnet,{Benjamin Corgnet},{Saint-Étienne},"{Left Wing, Center Attacking Midfield}",{AS Saint-Étienne},"{CAM, LW}"
3,2946.0,frederic guilbert,{Frédéric Guilbert},{Bordeaux},"{Right Back, Right Center Back}",{FC Girondins de Bordeaux},"{RB, CB}"
4,2948.0,nabil fekir,{Nabil Fekir},{Lyon},"{Right Center Forward, Left Center Forward, Ri...",{Olympique Lyonnais},"{ST, CAM, CF, LW, RW}"
5,2950.0,jonas martin,{Jonas Martin},{Montpellier},"{Left Defensive Midfield, Left Midfield, Right...",{Montpellier Hérault SC},"{CAM, LW, LM, CM, CDM}"
6,2953.0,abdou kader mangane,{Abdou Kader Mangane},{Gazélec Ajaccio},"{Left Center Back, Right Center Back, Center D...",{GFC Ajaccio},"{CDM, CB}"
7,2955.0,ronny rodelin,{Ronny Rodelin},"{Caen, Stade Malherbe Caen, Lille}","{Left Midfield, Right Midfield, Right Center F...","{LOSC Lille, Stade Malherbe Caen}","{RM, CAM, CF, CM, ST, LW, LM, RW}"
8,2956.0,bertrand isidore traore,{Bertrand Isidore Traoré},{Chelsea},"{Right Wing, Center Forward, Right Defensive M...",{Chelsea},"{ST, CF, CDM, RW}"
9,2959.0,julien feret,{Julien Féret},"{Caen, Stade Malherbe Caen}","{Left Defensive Midfield, Right Center Midfiel...",{Stade Malherbe Caen},"{CAM, CM, CDM}"


Validating Table

In [24]:
print("PLAYER MAPPING ATTRIBUTE VALIDATION")
print("-----------------------------------")
print("Rows in sb_players:", len(sb_players))

empty_team_mappings = sb_players["sofifa_teams"].apply(len).eq(0).sum()
empty_position_mappings = (sb_players["sofifa_positions"].apply(len).eq(0).sum())

print("Players with no mapped SoFIFA team:", empty_team_mappings)
print("Players with no mapped SoFIFA position:", empty_position_mappings)

# Show players with empty mapped positions, if any
position_mapping_issues = sb_players[sb_players["sofifa_positions"].apply(len).eq(0)][["player_id", "original_names", "positions", "sofifa_positions"]]

position_mapping_issues

PLAYER MAPPING ATTRIBUTE VALIDATION
-----------------------------------
Rows in sb_players: 1985
Players with no mapped SoFIFA team: 0
Players with no mapped SoFIFA position: 3


,player_id,original_names,positions,sofifa_positions
146,3262.0,{Łukasz Fabiański},{Goalkeeper},{}
1393,11508.0,{Łukasz Skorupski},{Goalkeeper},{}
1536,20066.0,{Darren Randolph},{Goalkeeper},{}


In [25]:
print("STATSBOMB PLAYER TABLE VALIDATION")
print("---------------------------------")
print("Unique StatsBomb player IDs:", shots["player_id"].nunique())
print("Rows in sb_players:", len(sb_players))
duplicate_ids = (sb_players["player_id"].duplicated(keep=False)) # Every StatsBomb player_id should occur exactly once
print("Duplicate player IDs in sb_players:", duplicate_ids.sum())

# Check whether one StatsBomb ID is associated with multiple different player names
id_name_counts = (shots[["player_id", "player_name_norm"]].drop_duplicates()
                  .groupby("player_id")["player_name_norm"].nunique())
conflicting_ids = id_name_counts[id_name_counts > 1]
print("Player IDs with multiple names:", len(conflicting_ids))

assert len(sb_players) == shots["player_id"].nunique(), ("sb_players does not contain exactly one row "
                                                         "per StatsBomb player_id." )
assert duplicate_ids.sum() == 0, ("Duplicate StatsBomb player IDs exist in sb_players.")
assert len(conflicting_ids) == 0, ("At least one StatsBomb player_id is associated "
                                   "with multiple normalized names.")

print("\n✓ StatsBomb player identity table is valid.")

STATSBOMB PLAYER TABLE VALIDATION
---------------------------------
Unique StatsBomb player IDs: 1985
Rows in sb_players: 1985
Duplicate player IDs in sb_players: 0
Player IDs with multiple names: 0

✓ StatsBomb player identity table is valid.


Inspecting Conflicting SB player_id

In [26]:
# conflicting_player_ids = conflicting_ids.index.tolist()
# conflicting_player_records = (shots[shots["player_id"].isin(conflicting_player_ids)][
#         ["player_id", "player", "player_name_norm", "team", "position"]]
#         .drop_duplicates().sort_values(["player_id", "player", "team", "position"]))

# conflicting_player_records

In [27]:
# (conflicting_player_records.groupby("player_id").agg(original_names=("player", lambda x: sorted(set(x))),
#                                                      normalized_names=("player_name_norm", lambda x: sorted(set(x))),
#                                                      teams=("team", lambda x: sorted(set(x))),
#                                                      positions=("position", lambda x: sorted(set(x)))))

Step 3B: Build and Validate SoFIFA Player Table

In [28]:
sofifa_players = df[["sofifa_id", "short_name", "long_name", "short_name_norm", "long_name_norm", "club_name", 
                     "player_positions", "position_set", "preferred_foot", "weak_foot", "shooting", "attacking_heading_accuracy"]].copy()

print("SOFIFA PLAYER TABLE VALIDATION")
print("------------------------------")
print("Rows in sofifa_players:", len(sofifa_players))
print("Unique sofifa_ids:", sofifa_players["sofifa_id"].nunique())
duplicate_sofifa_ids = (sofifa_players["sofifa_id"].duplicated(keep=False))
print("Duplicate sofifa_ids:", duplicate_sofifa_ids.sum())
empty_position_sets = (sofifa_players["position_set"].apply(len).eq(0))
print("Players with empty position_set:", empty_position_sets.sum())

assert len(sofifa_players) == sofifa_players["sofifa_id"].nunique(), ("SoFIFA table does not contain exactly one row per sofifa_id.")
assert duplicate_sofifa_ids.sum() == 0, ("Duplicate sofifa_ids exist in sofifa_players.")

print("\n✓ SoFIFA player candidate table is valid.")

SOFIFA PLAYER TABLE VALIDATION
------------------------------
Rows in sofifa_players: 13924
Unique sofifa_ids: 13924
Duplicate sofifa_ids: 0
Players with empty position_set: 0

✓ SoFIFA player candidate table is valid.


Step 4: Mapping Pipeline

In [29]:
def match_statsbomb_player(sb_row, sofifa_df):
    """
    Match one StatsBomb player to SoFIFA using:

    1. Exact normalized name
    2. Team
    3. Position

    Ambiguous cases remain unresolved.
    """

    # sb_name = sb_row["player_name_norm"]
    # sb_teams = sb_row["teams"]
    # sb_positions = sb_row["positions"]

    sb_name = sb_row["player_name_norm"]
    sb_teams = sb_row["sofifa_teams"]
    sb_positions = sb_row["sofifa_positions"]

    # STEP 1: NAME
    def name_tokens(name):
        if pd.isna(name):
            return set()

        return set(name.split())
    
    name_candidates = sofifa_df[(sofifa_df["long_name_norm"] == sb_name) | (sofifa_df["short_name_norm"] == sb_name)].copy()

    # No exact name match
    if len(name_candidates) == 0:
        team_candidates = sofifa_df[sofifa_df["club_name"].isin(sb_teams)].copy()
        sb_tokens = name_tokens(sb_name)

        token_candidates = []
        for _, candidate in team_candidates.iterrows():
            long_tokens = name_tokens(candidate["long_name_norm"])
            short_tokens = name_tokens(candidate["short_name_norm"])

            long_match = (sb_tokens.issubset(long_tokens) or long_tokens.issubset(sb_tokens))
            short_match = (sb_tokens.issubset(short_tokens) or short_tokens.issubset(sb_tokens))

            if long_match or short_match:
                token_candidates.append(candidate)

        # Exactly one deterministic same-team token candidate
        if len(token_candidates) == 1:
            candidate = token_candidates[0]
            position_overlap = sb_positions.intersection(candidate["position_set"])
            # If StatsBomb has mapped outfield positions, require position compatibility
            if sb_positions and not position_overlap:
                return {"status": "manual_review",
                        "match_method": "token_name_position_mismatch",
                        "sofifa_id": pd.NA,
                        "candidate_count": 1,
                        "candidate_ids": [candidate["sofifa_id"]]}
            return { "status": "matched",
                    "match_method": "team+token_name+position",
                    "sofifa_id": candidate["sofifa_id"],
                    "candidate_count": 1,
                    "candidate_ids": [candidate["sofifa_id"]]}
            # return {"status": "matched",
            #         "match_method": "team+token_name",
            #         "sofifa_id": candidate["sofifa_id"],
            #         "candidate_count": 1,
            #         "candidate_ids": [candidate["sofifa_id"]]}
        # Multiple plausible candidates: do not guess
        elif len(token_candidates) > 1:
            # Use position to disambiguate multiple same-team token-name candidates
            position_token_candidates = [candidate for candidate in token_candidates if (not sb_positions 
                                                                                         or bool(sb_positions.intersection(candidate["position_set"])
                                                                                                 )
                                                                                        )]
            # Position resolves ambiguity
            if len(position_token_candidates) == 1:
                candidate = position_token_candidates[0]
                return {"status": "matched",
                        "match_method": "team+token_name+position",
                        "sofifa_id": candidate["sofifa_id"],
                        "candidate_count": 1,
                        "candidate_ids": [candidate["sofifa_id"]]}
            # Multiple candidates still survive
            elif len(position_token_candidates) > 1:
                return {"status": "manual_review",
                        "match_method": "ambiguous_after_token_position",
                        "sofifa_id": pd.NA,
                        "candidate_count": len(position_token_candidates),
                        "candidate_ids": [candidate["sofifa_id"] for candidate in position_token_candidates]}

            # Token candidates with incompatible positions
            return { "status": "manual_review",
                    "match_method": "token_candidates_position_mismatch",
                    "sofifa_id": pd.NA,
                    "candidate_count": len(token_candidates),
                    "candidate_ids": [candidate["sofifa_id"] for candidate in token_candidates]}

        # Still nothing
        return {"status": "unmatched",
                "match_method": "no_name_match",
                "sofifa_id": pd.NA,
                "candidate_count": 0,
                "candidate_ids": []}
        
    # Exactly one name match
    elif len(name_candidates) == 1:
        candidate = name_candidates.iloc[0]
        return {"status": "matched", 
                "match_method": "name", 
                "sofifa_id": candidate["sofifa_id"],
                "candidate_count": 1,
                "candidate_ids": [candidate["sofifa_id"]]}
    # If len(name_candiadtes) > 1
    # STEP 2: TEAM
    team_candidates = name_candidates[name_candidates["club_name"].isin(sb_teams)].copy()

    # Exactly one candidate survives
    if len(team_candidates) == 1:
        candidate = team_candidates.iloc[0]
        return {"status": "matched",
                "match_method": "name+team",
                "sofifa_id": candidate["sofifa_id"],
                "candidate_count": 1,
                "candidate_ids": [candidate["sofifa_id"]]}
    # Important: If team gives ZERO candidates, I keep the original name candidates for manual inspection.
    elif len(team_candidates) == 0:
        return {"status": "manual_review",
                "match_method": "duplicate_name_team_mismatch",
                "sofifa_id": pd.NA,
                "candidate_count": len(name_candidates),
                "candidate_ids": name_candidates["sofifa_id"].tolist()}
    # If team_candidates > 1
    # STEP 3: POSITION
    def position_matches(position_set):
        return bool(sb_positions.intersection(position_set))

    position_candidates = team_candidates[team_candidates["position_set"].apply(position_matches)].copy()
    # Exactly one candidate survives
    if len(position_candidates) == 1:
        candidate = position_candidates.iloc[0]
        return {"status": "matched",
                "match_method": "name+team+position",
                "sofifa_id": candidate["sofifa_id"],
                "candidate_count": 1,
                "candidate_ids": [candidate["sofifa_id"]]}
    # STILL AMBIGUOUS
    elif len(position_candidates) > 1:
        return {"status": "manual_review",
                "match_method": "ambiguous_after_position",
                "sofifa_id": pd.NA,
                "candidate_count": len(position_candidates),
                "candidate_ids": position_candidates["sofifa_id"].tolist()}

    # Same name + same team, but position did not agree
    return {"status": "manual_review",
            "match_method": "position_mismatch",
            "sofifa_id": pd.NA,
            "candidate_count": len(team_candidates),
            "candidate_ids": team_candidates["sofifa_id"].tolist()}

TEST: CONTROLLED MATCHING SAMPLE

In [30]:
test_players = sb_players.sample(n=min(20, len(sb_players)),random_state=42)

test_results = []

for _, sb_player in test_players.iterrows():

    result = match_statsbomb_player(sb_player, sofifa_players)

    test_results.append({"player_id": sb_player["player_id"],
                         "statsbomb_name": sb_player["original_names"],
                         "statsbomb_teams": sb_player["sofifa_teams"],
                         "statsbomb_positions": sb_player["sofifa_positions"],
                         **result})

test_mapping = pd.DataFrame(test_results)
test_mapping

,player_id,statsbomb_name,statsbomb_teams,statsbomb_positions,status,match_method,sofifa_id,candidate_count,candidate_ids
0,5487.0,{Antoine Griezmann},{Atlético de Madrid},"{ST, RM, CF, LW, LM, RW}",matched,name,194765,1,[194765]
1,27054.0,{Roberto Lago Soto},{Getafe CF},{LB},matched,name,173600,1,[173600]
2,7473.0,{Lorenzo De Silvestri},{U.C. Sampdoria},"{RWB, RB}",matched,name,170320,1,[170320]
3,7695.0,{Samuel Souprayen},{Hellas Verona},"{LB, LWB}",matched,name,189155,1,[189155]
4,7010.0,{Gastón Exequiel Ramírez Pereyra},{Southampton},{RW},matched,name,201508,1,[201508]
5,3086.0,{Ben Davies},{Tottenham Hotspur},{LB},matched,name,151213,1,[151213]
6,4255.0,{Jean-Victor Makengo},{Stade Malherbe Caen},"{CM, CDM}",matched,name,230686,1,[230686]
7,6984.0,{Danilo D''Ambrosio},{Inter},"{LB, RB, RWB}",matched,name,198946,1,[198946]
8,7005.0,{João Pedro Cavaco Cancelo},{Valencia CF},"{LB, RB, RW}",matched,name,210514,1,[210514]
9,26009.0,{Roberto Soldado Rillo},{Villarreal CF},"{ST, CF}",matched,name,146758,1,[146758]


TEST: FIND DUPLICATE-NAME CASES

In [31]:
# Count how many SoFIFA players share each normalized long name
sofifa_name_counts = (sofifa_players.groupby("long_name_norm")["sofifa_id"].nunique())

duplicate_sofifa_names = set(sofifa_name_counts[sofifa_name_counts > 1].index)

# StatsBomb players whose name occurs multiple times in SoFIFA
duplicate_name_tests = (sb_players[sb_players["player_name_norm"].isin(duplicate_sofifa_names)].copy())

print("StatsBomb players with multiple SoFIFA name candidates: ", len(duplicate_name_tests))

duplicate_name_tests[["player_id", "original_names", "sofifa_teams", "sofifa_positions"]].head(20)

StatsBomb players with multiple SoFIFA name candidates:  5


,player_id,original_names,sofifa_teams,sofifa_positions
330,3616.0,{Callum Wilson},{AFC Bournemouth},"{ST, CF}"
528,4823.0,{Milan Gajić},{FC Girondins de Bordeaux},{RB}
825,6791.0,{Javier López Rodríguez},{RCD Espanyol de Barcelona},"{CAM, RB}"
1554,20527.0,{Jonas Olsson},{West Bromwich Albion},{CB}
1956,401455.0,{Mohamed Fofana},{Stade de Reims},{CB}


TEST: DUPLICATE-NAME DISAMBIGUATION

In [32]:
duplicate_test_results = []

for _, sb_player in duplicate_name_tests.iterrows():

    result = match_statsbomb_player(sb_player,sofifa_players)

    duplicate_test_results.append({"player_id": sb_player["player_id"],
                                   "statsbomb_name": sb_player["original_names"],
                                   "statsbomb_teams": sb_player["sofifa_teams"],
                                   "statsbomb_positions": sb_player["sofifa_positions"],
                                   **result})

duplicate_test_mapping = pd.DataFrame(duplicate_test_results)

duplicate_test_mapping

,player_id,statsbomb_name,statsbomb_teams,statsbomb_positions,status,match_method,sofifa_id,candidate_count,candidate_ids
0,3616.0,{Callum Wilson},{AFC Bournemouth},"{ST, CF}",matched,name+team,196978,1,[196978]
1,4823.0,{Milan Gajić},{FC Girondins de Bordeaux},{RB},matched,name+team,229782,1,[229782]
2,6791.0,{Javier López Rodríguez},{RCD Espanyol de Barcelona},"{CAM, RB}",matched,name+team,195361,1,[195361]
3,20527.0,{Jonas Olsson},{West Bromwich Albion},{CB},matched,name+team,45842,1,[45842]
4,401455.0,{Mohamed Fofana},{Stade de Reims},{CB},matched,name+team,173033,1,[173033]


RUN FULL PLAYER MAPPING

In [33]:
mapping_results = []

for _, sb_player in sb_players.iterrows():
    result = match_statsbomb_player(sb_player, sofifa_players)
    mapping_results.append({"player_id": sb_player["player_id"],
                            "statsbomb_name": sb_player["original_names"],
                            "statsbomb_name_norm": sb_player["player_name_norm"],
                            "statsbomb_teams": sb_player["sofifa_teams"],
                            "statsbomb_positions": sb_player["sofifa_positions"],
                            **result})

player_mapping = pd.DataFrame(mapping_results)
player_mapping.head(20)

,player_id,statsbomb_name,statsbomb_name_norm,statsbomb_teams,statsbomb_positions,status,match_method,sofifa_id,candidate_count,candidate_ids
0,2936.0,{Christophe Kerbrat},christophe kerbrat,{En Avant de Guingamp},{CB},matched,name,204338,1,[204338]
1,2943.0,{Lucas Deaux},lucas deaux,{FC Nantes},"{CDM, CB}",matched,name,178051,1,[178051]
2,2944.0,{Benjamin Corgnet},benjamin corgnet,{AS Saint-Étienne},"{CAM, LW}",matched,name,200966,1,[200966]
3,2946.0,{Frédéric Guilbert},frederic guilbert,{FC Girondins de Bordeaux},"{RB, CB}",matched,name,227222,1,[227222]
4,2948.0,{Nabil Fekir},nabil fekir,{Olympique Lyonnais},"{ST, CAM, CF, LW, RW}",matched,name,216594,1,[216594]
5,2950.0,{Jonas Martin},jonas martin,{Montpellier Hérault SC},"{CAM, LW, LM, CM, CDM}",matched,name,197813,1,[197813]
6,2953.0,{Abdou Kader Mangane},abdou kader mangane,{GFC Ajaccio},"{CDM, CB}",matched,name,47860,1,[47860]
7,2955.0,{Ronny Rodelin},ronny rodelin,"{LOSC Lille, Stade Malherbe Caen}","{RM, CAM, CF, CM, ST, LW, LM, RW}",matched,team+token_name+position,188397,1,[188397]
8,2956.0,{Bertrand Isidore Traoré},bertrand isidore traore,{Chelsea},"{ST, CF, CDM, RW}",matched,name,207948,1,[207948]
9,2959.0,{Julien Féret},julien feret,{Stade Malherbe Caen},"{CAM, CM, CDM}",matched,name,163804,1,[163804]


Auditing Results

In [34]:
print("PLAYER MAPPING RESULTS")
print("----------------------")
print("Total StatsBomb players:", len(player_mapping), "\n")

print(player_mapping["status"].value_counts(dropna=False), "\n")
print(player_mapping["match_method"].value_counts(dropna=False), "\n")
print(player_mapping["status"].value_counts(normalize=True, dropna=False).mul(100).round(2))

PLAYER MAPPING RESULTS
----------------------
Total StatsBomb players: 1985 

status
matched          1803
unmatched         147
manual_review      35
Name: count, dtype: int64 

match_method
name                                  1671
no_name_match                          147
team+token_name+position               127
token_name_position_mismatch            24
ambiguous_after_token_position           8
name+team                                5
token_candidates_position_mismatch       3
Name: count, dtype: int64 

status
matched          90.83
unmatched         7.41
manual_review     1.76
Name: proportion, dtype: float64


INSPECT UNMATCHED PLAYERS

In [35]:
unmatched_players = (player_mapping[player_mapping["status"] == "unmatched"].copy())

print("UNMATCHED PLAYER AUDIT")
print("----------------------")
print("Total unmatched:", len(unmatched_players))
percentage_unmatched = len(unmatched_players) / len(player_mapping) * 100
print("Unmatched percentage: ", round(percentage_unmatched, 2), "%")

unmatched_players[["player_id", "statsbomb_name", "statsbomb_name_norm", "statsbomb_teams", "statsbomb_positions", "match_method"]].head(50)

UNMATCHED PLAYER AUDIT
----------------------
Total unmatched: 147
Unmatched percentage:  7.41 %


,player_id,statsbomb_name,statsbomb_name_norm,statsbomb_teams,statsbomb_positions,match_method
29,3009.0,{Kylian Mbappé Lottin},kylian mbappe lottin,{AS Monaco},"{RM, RW}",no_name_match
33,3020.0,{Rodrigue Casimir Ninga},rodrigue casimir ninga,{Montpellier Hérault SC},"{ST, CF, LW, LM, CDM, RW}",no_name_match
37,3025.0,{Christopher Nkunku},christopher nkunku,{Paris Saint-Germain},{CM},no_name_match
82,3109.0,{Malcom Filipe Silva de Oliveira},malcom filipe silva de oliveira,{FC Girondins de Bordeaux},"{LW, LM, RW}",no_name_match
93,3141.0,{André-Frank Zambo Anguissa},andre frank zambo anguissa,{Olympique de Marseille},"{CAM, CM, CDM}",no_name_match
101,3162.0,{Gnaly Maxwell Cornet},gnaly maxwell cornet,{Olympique Lyonnais},"{ST, CF, LW, LM, CM, RW}",no_name_match
111,3186.0,{Julian Draxler},julian draxler,{VfL Wolfsburg},{CAM},no_name_match
114,3191.0,{Ramy Bensebaini},ramy bensebaini,{Montpellier Hérault SC},"{LB, CDM, CB}",no_name_match
126,3217.0,{Jemerson de Jesus Nascimento},jemerson de jesus nascimento,{AS Monaco},{RB},no_name_match
127,3218.0,{Marcos Paulo Mesquita Lopes},marcos paulo mesquita lopes,"{AS Monaco, LOSC Lille}","{CAM, LW, RW}",no_name_match


In [36]:
print("UNMATCHED PLAYERS BY STATSBOMB TEAM")
print("-----------------------------------")

unmatched_team_counts = (unmatched_players["statsbomb_teams"].astype(str).value_counts())
unmatched_team_counts.head(30)

UNMATCHED PLAYERS BY STATSBOMB TEAM
-----------------------------------


statsbomb_teams
{'Stade Rennais FC'}            6
{'ESTAC Troyes'}                5
{'West Ham United'}             5
{'Sevilla FC'}                  4
{'FC Girondins de Bordeaux'}    4
{'LOSC Lille'}                  4
{'Olympique Lyonnais'}          4
{'SC Bastia'}                   4
{'Empoli'}                      4
{'Norwich City'}                4
{'Manchester United'}           4
{'SD Eibar'}                    3
{'OGC Nice'}                    3
{'Villarreal CF'}               3
{'Roma'}                        3
{'Udinese Calcio'}              3
{'Liverpool'}                   3
{'Aston Villa'}                 3
{'Palermo'}                     3
{'FC Nantes'}                   3
{'AS Monaco'}                   3
{'SV Werder Bremen'}            3
{'West Bromwich Albion'}        3
{'Olympique de Marseille'}      3
{'Chelsea'}                     3
{'Stade de Reims'}              2
{'Granada CF'}                  2
{'Real Sporting de Gijón'}      2
{'VfL Wolfsburg'}               

TEST UNMATCHED PLAYERS AGAINST SOFIFA SHORT NAMES

In [37]:
def count_short_name_candidates(sb_name):
    """ Count exact normalized SoFIFA short-name matches for one StatsBomb normalized player name. """

    candidates = sofifa_players[sofifa_players["short_name_norm"] == sb_name]

    return len(candidates)


unmatched_players["short_name_candidate_count"] = (unmatched_players["statsbomb_name_norm"].apply(count_short_name_candidates))


print("SHORT-NAME DIAGNOSTIC")
print("---------------------\n")
print(unmatched_players["short_name_candidate_count"].value_counts().sort_index())

short_name_matches = unmatched_players[ unmatched_players["short_name_candidate_count"] > 0][["player_id", "statsbomb_name", "statsbomb_name_norm", "statsbomb_teams", "short_name_candidate_count"]]

print("\nUnmatched StatsBomb players with at least one exact SoFIFA short-name candidate:",len(short_name_matches))

short_name_matches.head(50)

SHORT-NAME DIAGNOSTIC
---------------------

short_name_candidate_count
0    147
Name: count, dtype: int64

Unmatched StatsBomb players with at least one exact SoFIFA short-name candidate: 0


,player_id,statsbomb_name,statsbomb_name_norm,statsbomb_teams,short_name_candidate_count


INSPECT SAME-TEAM SOFIFA CANDIDATES

In [38]:
sample_unmatched = unmatched_players.head(20)

same_team_candidates = []

for _, sb_player in sample_unmatched.iterrows():
    # Get this player's StatsBomb identity row
    sb_identity = sb_players[sb_players["player_id"] == sb_player["player_id"]].iloc[0]
    teams = sb_identity["sofifa_teams"]

    # Find every SoFIFA player belonging to those clubs
    candidates = sofifa_players[sofifa_players["club_name"].isin(teams)]
    same_team_candidates.append({"player_id": sb_player["player_id"],
                                 "statsbomb_name": sb_player["statsbomb_name"],
                                 "statsbomb_name_norm": sb_player["statsbomb_name_norm"],
                                 "sofifa_team": teams,
                                 "candidate_names": candidates[["sofifa_id", "short_name", "long_name", "player_positions"]].to_dict("records")})

same_team_audit = pd.DataFrame(same_team_candidates)
same_team_audit

,player_id,statsbomb_name,statsbomb_name_norm,sofifa_team,candidate_names
0,3009.0,{Kylian Mbappé Lottin},kylian mbappe lottin,{AS Monaco},"[{'sofifa_id': 162347, 'short_name': 'João Mou..."
1,3020.0,{Rodrigue Casimir Ninga},rodrigue casimir ninga,{Montpellier Hérault SC},"[{'sofifa_id': 153260, 'short_name': 'Hilton',..."
2,3025.0,{Christopher Nkunku},christopher nkunku,{Paris Saint-Germain},"[{'sofifa_id': 41236, 'short_name': 'Z. Ibrahi..."
3,3109.0,{Malcom Filipe Silva de Oliveira},malcom filipe silva de oliveira,{FC Girondins de Bordeaux},"[{'sofifa_id': 192333, 'short_name': 'L. Sané'..."
4,3141.0,{André-Frank Zambo Anguissa},andre frank zambo anguissa,{Olympique de Marseille},"[{'sofifa_id': 188829, 'short_name': 'N. Nkoul..."
5,3162.0,{Gnaly Maxwell Cornet},gnaly maxwell cornet,{Olympique Lyonnais},"[{'sofifa_id': 193301, 'short_name': 'A. Lacaz..."
6,3186.0,{Julian Draxler},julian draxler,{VfL Wolfsburg},"[{'sofifa_id': 171919, 'short_name': 'Naldo', ..."
7,3191.0,{Ramy Bensebaini},ramy bensebaini,{Montpellier Hérault SC},"[{'sofifa_id': 153260, 'short_name': 'Hilton',..."
8,3217.0,{Jemerson de Jesus Nascimento},jemerson de jesus nascimento,{AS Monaco},"[{'sofifa_id': 162347, 'short_name': 'João Mou..."
9,3218.0,{Marcos Paulo Mesquita Lopes},marcos paulo mesquita lopes,"{AS Monaco, LOSC Lille}","[{'sofifa_id': 162347, 'short_name': 'João Mou..."


TEST TOKEN-BASED NAME CONTAINMENT WITHIN SAME TEAM

In [39]:
# def name_tokens(name):
#     """Convert a normalized name into a set of name tokens."""
                                
#     if pd.isna(name):
#         return set()

#     return set(name.split())

# token_match_results = []

# for _, sb_player in unmatched_players.iterrows():
#     sb_identity = sb_players[sb_players["player_id"] == sb_player["player_id"]].iloc[0]
#     sb_name = sb_player["statsbomb_name_norm"]
#     sb_tokens = name_tokens(sb_name)

#     teams = sb_identity["sofifa_teams"]

#     team_candidates = sofifa_players[sofifa_players["club_name"].isin(teams)].copy()

#     candidate_matches = []
#     for _, candidate in team_candidates.iterrows():
#         long_tokens = name_tokens(candidate["long_name_norm"])
#         short_tokens = name_tokens(candidate["short_name_norm"])

#         long_match = (sb_tokens.issubset(long_tokens) or long_tokens.issubset(sb_tokens))
#         short_match = (sb_tokens.issubset(short_tokens) or short_tokens.issubset(sb_tokens))

#         if long_match or short_match:
#             candidate_matches.append({"sofifa_id": candidate["sofifa_id"],
#                                       "short_name": candidate["short_name"],
#                                       "long_name": candidate["long_name"],
#                                       "club_name": candidate["club_name"],
#                                       "player_positions": candidate["player_positions"]})

#     token_match_results.append({"player_id": sb_player["player_id"],
#                                 "statsbomb_name": sb_player["statsbomb_name"],
#                                 "statsbomb_name_norm": sb_name,
#                                 "statsbomb_teams": teams,
#                                 "token_candidate_count": len(candidate_matches),
#                                 "token_candidates": candidate_matches})


# token_match_audit = pd.DataFrame(token_match_results)

# print("TOKEN-CONTAINMENT DIAGNOSTIC")
# print("----------------------------")
# print(token_match_audit["token_candidate_count"].value_counts().sort_index())
# print("\nPlayers with at least one token candidate:", (token_match_audit["token_candidate_count"] > 0).sum())

# token_match_audit[token_match_audit["token_candidate_count"] > 0].head(30)

STEP 6: BUILD MANUAL REVIEW TABLE

In [40]:
unresolved = player_mapping[
    player_mapping["status"].isin(
        ["unmatched", "manual_review"]
    )
].copy()


manual_review_rows = []

for _, row in unresolved.iterrows():

    # Get the corresponding StatsBomb identity information
    sb_row = sb_players[
        sb_players["player_id"] == row["player_id"]
    ].iloc[0]

    # Search only the clubs this player represented
    team_candidates = sofifa_players[
        sofifa_players["club_name"].isin(
            sb_row["sofifa_teams"]
        )
    ].copy()

    manual_review_rows.append({
        "player_id": row["player_id"],
        "statsbomb_name": row["statsbomb_name"],
        "statsbomb_name_norm": row["statsbomb_name_norm"],

        "teams": sb_row["teams"],
        "mapped_teams": sb_row["sofifa_teams"],

        "positions": sb_row["positions"],
        "mapped_positions": sb_row["sofifa_positions"],

        "current_status": row["status"],
        "current_match_method": row["match_method"],

        # Existing candidates from the matcher, if any
        "candidate_ids": row["candidate_ids"],

        # Full same-team roster for manual resolution
        "team_candidates": team_candidates[
            [
                "sofifa_id",
                "short_name",
                "long_name",
                "club_name",
                "player_positions"
            ]
        ].to_dict("records"),

        # Fill this manually
        "manual_sofifa_id": pd.NA
    })


manual_review = pd.DataFrame(
    manual_review_rows
)

print("Players requiring manual resolution:", len(manual_review))

manual_review.head(20)

Players requiring manual resolution: 182


,player_id,statsbomb_name,statsbomb_name_norm,teams,mapped_teams,positions,mapped_positions,current_status,current_match_method,candidate_ids,team_candidates,manual_sofifa_id
0,3009.0,{Kylian Mbappé Lottin},kylian mbappe lottin,{AS Monaco},{AS Monaco},"{Right Wing, Right Midfield}","{RM, RW}",unmatched,no_name_match,[],"[{'sofifa_id': 162347, 'short_name': 'João Mou...",<NA>
1,3020.0,{Rodrigue Casimir Ninga},rodrigue casimir ninga,{Montpellier},{Montpellier Hérault SC},"{Left Defensive Midfield, Left Midfield, Right...","{ST, CF, LW, LM, CDM, RW}",unmatched,no_name_match,[],"[{'sofifa_id': 153260, 'short_name': 'Hilton',...",<NA>
2,3025.0,{Christopher Nkunku},christopher nkunku,{Paris Saint-Germain},{Paris Saint-Germain},{Left Center Midfield},{CM},unmatched,no_name_match,[],"[{'sofifa_id': 41236, 'short_name': 'Z. Ibrahi...",<NA>
3,3053.0,{Leroy Sané},leroy sane,{Schalke 04},{FC Schalke 04},{Right Wing},{RW},manual_review,token_candidates_position_mismatch,"[191541, 222492]","[{'sofifa_id': 179784, 'short_name': 'B. Höwed...",<NA>
4,3083.0,{Heung-Min Son},heung min son,"{Tottenham Hotspur, Bayer Leverkusen}","{Bayer 04 Leverkusen, Tottenham Hotspur}","{Left Defensive Midfield, Right Wing, Right De...","{CAM, LW, CDM, RW}",manual_review,token_name_position_mismatch,[221383],"[{'sofifa_id': 190460, 'short_name': 'C. Eriks...",<NA>
5,3109.0,{Malcom Filipe Silva de Oliveira},malcom filipe silva de oliveira,{Bordeaux},{FC Girondins de Bordeaux},"{Left Midfield, Left Wing, Right Wing}","{LW, LM, RW}",unmatched,no_name_match,[],"[{'sofifa_id': 192333, 'short_name': 'L. Sané'...",<NA>
6,3141.0,{André-Frank Zambo Anguissa},andre frank zambo anguissa,{Olympique de Marseille},{Olympique de Marseille},"{Left Defensive Midfield, Right Center Midfiel...","{CAM, CM, CDM}",unmatched,no_name_match,[],"[{'sofifa_id': 188829, 'short_name': 'N. Nkoul...",<NA>
7,3162.0,{Gnaly Maxwell Cornet},gnaly maxwell cornet,{Lyon},{Olympique Lyonnais},"{Right Center Midfield, Left Midfield, Right C...","{ST, CF, LW, LM, CM, RW}",unmatched,no_name_match,[],"[{'sofifa_id': 193301, 'short_name': 'A. Lacaz...",<NA>
8,3186.0,{Julian Draxler},julian draxler,{Wolfsburg},{VfL Wolfsburg},{Center Attacking Midfield},{CAM},unmatched,no_name_match,[],"[{'sofifa_id': 171919, 'short_name': 'Naldo', ...",<NA>
9,3191.0,{Ramy Bensebaini},ramy bensebaini,{Montpellier},{Montpellier Hérault SC},"{Left Defensive Midfield, Center Defensive Mid...","{LB, CDM, CB}",unmatched,no_name_match,[],"[{'sofifa_id': 153260, 'short_name': 'Hilton',...",<NA>


In [41]:
from rapidfuzz import fuzz

def rank_manual_candidates(row, top_n=5):
    """ Rank same-team SoFIFA candidates for manual inspection.
    IMPORTANT: This does NOT automatically accept any match. It only helps us inspect the most likely candidates first. """

    sb_name = row["statsbomb_name_norm"]
    sb_positions = row["mapped_positions"]

    ranked = []
    for candidate in row["team_candidates"]:
        candidate_id = candidate["sofifa_id"]

        # Get full candidate row from validated SoFIFA table
        fifa_row = sofifa_players[sofifa_players["sofifa_id"] == candidate_id].iloc[0]

        # Compare against both SoFIFA long and short names
        long_score = fuzz.WRatio(sb_name,fifa_row["long_name_norm"])
        short_score = fuzz.WRatio(sb_name,fifa_row["short_name_norm"])
        name_score = max(long_score, short_score)

        # Check position compatibility
        position_overlap = (sb_positions.intersection(fifa_row["position_set"]))

        ranked.append({"sofifa_id": candidate_id,
                       "short_name": fifa_row["short_name"],
                       "long_name": fifa_row["long_name"],
                       "club_name": fifa_row["club_name"],
                       "player_positions": fifa_row["player_positions"],
                       "name_score": round(name_score, 1),
                       "position_overlap": position_overlap})

    # Highest name similarity first
    ranked = sorted(ranked, key=lambda x: x["name_score"],reverse=True)

    return ranked[:top_n]

import numpy as np
cols = ["sofifa_id", "short_name", "long_name", "player_positions", "club_name", "shooting", "attacking_heading_accuracy", "preferred_foot", "weak_foot"] 
manual_review[cols] = np.nan
manual_review[["player_id", "statsbomb_name", "statsbomb_name_norm", "sofifa_id", "short_name", "long_name", "player_positions", "club_name", "shooting",
               "attacking_heading_accuracy", "preferred_foot", "weak_foot"]].to_csv("/Users/nanakwamekankam/Desktop/Football_Stats/xG/datasets/processed/manual_review.csv", index=False)

# manual_review["ranked_candidates"] = ( manual_review.apply(rank_manual_candidates, axis=1))

# manual_review[["player_id", "statsbomb_name", "mapped_teams", "mapped_positions", "current_match_method",
#                "ranked_candidates", "manual_sofifa_id"]].head(20)


In [42]:
name_existence_results = []

for _, player in unresolved.iterrows():
    sb_name = player["statsbomb_name_norm"]
    long_match = sofifa_players["long_name_norm"].eq(sb_name)
    short_match = sofifa_players["short_name_norm"].eq(sb_name)
    matches = sofifa_players[long_match | short_match]

    name_existence_results.append({"player_id": player["player_id"],
                                   "statsbomb_name": player["statsbomb_name"],
                                   "statsbomb_name_norm": sb_name,
                                   "exists_in_sofifa": len(matches) > 0,
                                   "match_count": len(matches), 
                                   "sofifa_matches": matches[["sofifa_id", "short_name", "long_name", "club_name"]].to_dict("records") })

name_existence = pd.DataFrame(name_existence_results)

print("Total unresolved:", len(name_existence))
print(name_existence["exists_in_sofifa"].value_counts(dropna=False))

Total unresolved: 182
exists_in_sofifa
False    182
Name: count, dtype: int64


MANUAL OVERRIDES

In [43]:
# cols = ["sofifa_id", "statsbomb_id", "short_name", "long_name", "player_positions", "club_name", "shooting", "attacking_heading_accuracy", "preferred_foot", "weak_foot"]
# missing_player_attributes = manual_review[cols].copy()


# pd.DataFrame([{
#     "sofifa_id": 231747,  # no SoFIFA ID because player is absent
#     "statsbomb_id": 3009,
#     "short_name": "K. Mbappe Lottin",
#     "long_name": "Kylian Mbappé Lottin",
#     "player_positions": "LW",
#     "club_name": "AS Monaco",
#     "shooting": AS Monaco,
#     "attacking_heading_accuracy": 41,
#     "preferred_foot": "Right",
#     "weak_foot": 3
#     },

#     {"sofifa_id": 231747,  # no SoFIFA ID because player is absent
#     "statsbomb_id": 3020.0,
#     "short_name": "K. Mbappe Lottin",
#     "long_name": "Rodrigue Casimir Ning",
#     "player_positions": "ST, RW",
#     "club_name": "AS Monaco",
#     "shooting": 60,
#     "attacking_heading_accuracy": 41,
#     "preferred_foot": "Right",
#     "weak_foot": 3
#         },
    

#     # add remaining players here
# ])

In [44]:
# missing_player_attributes["long_name_norm"] = (missing_player_attributes["long_name"].apply(normalize_name))
# missing_player_attributes["short_name_norm"] = (missing_player_attributes["short_name"].apply(normalize_name))
# missing_player_attributes["position_set"] = (missing_player_attributes["player_positions"].apply(parse_sofifa_positions))

In [45]:
# sofifa_players_complete = pd.concat([sofifa_players, missing_player_attributes], ignore_index=True)

Final Dataset

In [46]:
# shots_final = pd.merge(sofifa_players_complete, shots, on="sofifa_id", how="left")

In [47]:
# assert shots_final["shooting"].isna().sum() == 0
# assert shots_final["attacking_heading_accuracy"].isna().sum() == 0
# assert shots_final["preferred_foot"].isna().sum() == 0
# assert shots_final["weak_foot"].isna().sum() == 0

Temporary Dataset: Not featuring the missing players

In [ ]:
# 1. Keep ONLY successfully matched StatsBomb players
matched_players = (player_mapping[player_mapping["status"] == "matched"][["player_id", "sofifa_id", "match_method"]].copy())
print("Matched players:", len(matched_players))

player_attributes = sofifa_players[["sofifa_id", "shooting", "attacking_heading_accuracy", "preferred_foot", "weak_foot"]].copy()
matched_player_attributes = matched_players.merge(player_attributes, on="sofifa_id", how="inner", validate="many_to_one")

# Add those attributes to every shot.
# INNER JOIN to intentionally remove shots from players that were unresolved / absent from players_16.
shots_with_player_attributes = shots.merge(matched_player_attributes, on="player_id", how="inner", validate="many_to_one")
shots_with_player_attributes.head()

Matched players: 1803


,location,player,player_id,position,shot_aerial_won,shot_first_time,shot_statsbomb_xg,team,under_pressure,shot_open_goal,...,angle,player_name_norm,sofifa_club,sofifa_position,sofifa_id,match_method,shooting,attacking_heading_accuracy,preferred_foot,weak_foot
0,"[94.5, 42.9]",Aaron Ramsey,3517.0,Right Wing,0,1,0.038832,Arsenal,0,0,...,17.611035,aaron ramsey,Arsenal,{RW},186561,name,78.0,58,Right,3
1,"[93.5, 48.4]",Francesc Fàbregas i Soler,3478.0,Right Defensive Midfield,0,0,0.031541,Chelsea,1,0,...,15.648789,francesc fabregas i soler,Chelsea,{CDM},162895,name,78.0,74,Right,3
2,"[89.8, 22.3]",Alexis Alejandro Sánchez Sánchez,3385.0,Left Wing,0,0,0.006660,Arsenal,1,0,...,11.297814,alexis alejandro sanchez sanchez,Arsenal,{LW},184941,name,83.0,60,Right,3
3,"[98.2, 26.0]",Diego da Silva Costa,5198.0,Center Forward,0,0,0.022902,Chelsea,0,0,...,14.904419,diego da silva costa,Chelsea,"{ST, CF}",179844,name,83.0,82,Right,4
4,"[105.7, 24.2]",Theo Walcott,3668.0,Center Forward,0,0,0.049741,Arsenal,0,0,...,14.633756,theo walcott,Arsenal,"{ST, CF}",164859,name,77.0,53,Right,3


In [50]:
shots_with_player_attributes.shape

(36236, 42)

In [51]:
shots_with_player_attributes.to_csv("/Users/nanakwamekankam/Desktop/Football_Stats/xG/datasets/processed/shots_v1.csv", index=False)